# Qiskit Fall Fest 2026 — Challenge 3: Can you change the correlation?
**IBM Quantum | Vishnu Institute of Technology**  
**Name:** Srikar

## Objective
Start from the Bell-state circuit, add **one** gate to change the correlation between the two qubits, predict the outcome, run both circuits for 100 shots, and compare the results with histograms.

## 1. Setup

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib pylatexenc

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

SHOTS = 100
simulator = AerSimulator()

def run(qc, shots=SHOTS):
    """Execute a circuit on the Aer simulator and return the counts."""
    return simulator.run(transpile(qc, simulator), shots=shots).result().get_counts()

def report(name, counts):
    same = counts.get('00', 0) + counts.get('11', 0)
    diff = counts.get('01', 0) + counts.get('10', 0)
    print(f'{name}: {dict(sorted(counts.items()))}')
    print(f'   same (00 + 11) = {same}, different (01 + 10) = {diff}')

print('Setup complete. Shots per experiment:', SHOTS)

## 2. Original circuit — Bell state
Gates: `H(0)`, `CX(0,1)` — 2 gates before measurement.

In [ ]:
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)          # superposition on qubit 0
qc_bell.cx(0, 1)      # entangle qubit 1 with qubit 0
qc_bell.measure([0, 1], [0, 1])

print('Original Bell circuit (2 gates before measurement):')
display(qc_bell.draw('mpl'))

In [ ]:
counts_bell = run(qc_bell)
report('Original Bell circuit', counts_bell)
plot_histogram(counts_bell, title='Original Bell circuit (100 shots)')
plt.show()

## 3. Modified circuit — add one `X` gate on qubit 1

**Modification:** add an `X` gate on qubit 1 right after the CNOT.

### Prediction (before execution)
- Original Bell state: (|00⟩ + |11⟩)/√2 → only `00` and `11`, about 50 each.
- After `X` on qubit 1 the state becomes (|01⟩ + |10⟩)/√2.
- So the two qubits should now **always give different results**: only `01` and `10`, about 50 each, and `00` and `11` should not appear.
- The qubits are still perfectly linked, but the correlation changes from "same" to "opposite".

**Think before you run:** the counts will not be exactly 50/50, because only 100 shots are measured.

In [ ]:
qc_mod = QuantumCircuit(2, 2)
qc_mod.h(0)
qc_mod.cx(0, 1)
qc_mod.x(1)           # <-- the one added gate
qc_mod.measure([0, 1], [0, 1])

print('Modified circuit (3 gates before measurement, limit is 4):')
display(qc_mod.draw('mpl'))

In [ ]:
counts_mod = run(qc_mod)
report('Modified circuit (H, CX, X)', counts_mod)
plot_histogram(counts_mod, title='Modified circuit: Bell + X on qubit 1 (100 shots)')
plt.show()

## 4. Compare the original and modified circuits
Both results on the same axes, with all four outcomes shown (missing outcomes count as 0).

In [ ]:
labels = ["00", "01", "10", "11"]
orig = [counts_bell.get(k, 0) for k in labels]
mod = [counts_mod.get(k, 0) for k in labels]

x = list(range(len(labels)))
w = 0.38
fig, ax = plt.subplots(figsize=(8, 4.5))
bars_o = ax.bar([i - w / 2 for i in x], orig, w, label="Original (H, CX)")
bars_m = ax.bar([i + w / 2 for i in x], mod, w, label="Modified (H, CX, X)")
ax.bar_label(bars_o)
ax.bar_label(bars_m)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Measured outcome (q1 q0)")
ax.set_ylabel("Counts out of 100 shots")
ax.set_title("Original vs modified circuit (100 shots each)")
ax.legend()
plt.show()

In [ ]:
print(f"{'Outcome':<10}{'Original':>10}{'Modified':>10}")
for k, co, cm in zip(labels, orig, mod):
    print(f"{k:<10}{co:>10}{cm:>10}")

for name, c in [("Original", counts_bell), ("Modified", counts_mod)]:
    same = c.get("00", 0) + c.get("11", 0)
    diff = c.get("01", 0) + c.get("10", 0)
    print(f"\n{name}: same-bit outcomes (00+11) = {same}, different-bit outcomes (01+10) = {diff}")

## 5. Observed change and explanation (edit after running)

| | Original | Modified |
|---|---|---|
| Expected outcomes | `00`, `11` | `01`, `10` |
| Actual (fill in) | 00: ___, 11: ___ | 01: ___, 10: ___ |

**Explanation:** The Hadamard and CNOT gates create the Bell state (|00⟩ + |11⟩)/√2, so both qubits always agree. Adding an `X` gate on qubit 1 flips that qubit's value in every branch of the superposition: |00⟩ becomes |01⟩ and |11⟩ becomes |10⟩. The state is now (|01⟩ + |10⟩)/√2, so the qubits always disagree. Each qubit on its own is still random (about 50% `0` and 50% `1`), and the pair is still perfectly linked, only now anti-correlated. The counts are not exactly 50/50 because only 100 shots are measured.

*Note:* Qiskit prints bit strings as `q1 q0` (qubit 0 is the rightmost bit). This does not affect the conclusion here because both allowed outcomes in each circuit are symmetric.